In [1]:
import torch
from pathlib import Path
import os
import json
import pandas as pd
from tqdm import tqdm
from model import DiagnosticModel
from train_utils import get_info
from collections import defaultdict
import numpy as np

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
results_dirs = [Path('../outputs4')]
all_runs = []
for results_dir in results_dirs:
    all_runs.extend([results_dir / sbdir for sbdir in os.listdir(results_dir)])
device = 'cuda'

print(all_runs)

[PosixPath('../outputs3/b_mm_mask_norm2'), PosixPath('../outputs3/b_mm'), PosixPath('../outputs3/b_mm_mask_norm'), PosixPath('../outputs3/b_mm_mask_norm_perc2'), PosixPath('../outputs3/b_mm_mask_norm_perc'), PosixPath('../outputs3/temp'), PosixPath('../outputs3/b_mm2'), PosixPath('../outputs/b_mask2'), PosixPath('../outputs/balance_w'), PosixPath('../outputs/balance_o_download'), PosixPath('../outputs/balance_w_download2'), PosixPath('../outputs/o_mask2'), PosixPath('../outputs/balance_w_download'), PosixPath('../outputs/o_stack'), PosixPath('../outputs/b_stack'), PosixPath('../outputs/balance_b_download'), PosixPath('../outputs/balance_b_download2'), PosixPath('../outputs/balance_o'), PosixPath('../outputs/o_mask'), PosixPath('../outputs/balance_b'), PosixPath('../outputs/b_stack2'), PosixPath('../outputs/o_stack2'), PosixPath('../outputs/balance_o_download2'), PosixPath('../outputs/b_mask')]


In [ ]:
results_dict = defaultdict(list)

for trial in all_runs:
    for sbdir_path in all_runs:
        sbdir = sbdir_path.stem

        model_path = sbdir_path / 'best_model.pth'
        params_path = sbdir_path / 'params.json'
        results_path = sbdir_path / 'test_results.json'

        with open(params_path, 'r') as f:
            params = json.load(f)

        if not os.path.exists(results_path):
            continue 
        with open(results_path, 'r') as f:
            results = json.load(f)

        results_dict['exps'].append(sbdir)
        results_dict['auc'].append(float(results['auc']))
        # results_dict['val_metric'].append(float(best_val_metric_score)) # ignore b/c sometimes f1 and sometimes acc

        for test_metric, test_value in get_info(np.array(results['val_cfvalues'])).items():
            results_dict[f'test_{test_metric}'].append(test_value)
    


100%|██████████| 24/24 [00:00<00:00, 92.13it/s]


In [4]:
sort_metric = ['auc', 'test_acc']

df = pd.DataFrame(results_dict)
df = df[['exps'] + sorted([col for col in df.columns if col != 'exps'])]

df = df.sort_values(by=sort_metric, ascending=False)
df = df.reset_index(drop=True)
df


,exps,auc,test_acc,test_f1,test_fn,test_fp,test_fpr,test_prec,test_recall,test_tn,test_tp,test_tpr
0,b_mm_mask_norm_perc2,0.869741,0.926916,0.506024,0.048128,0.024955,0.027290,0.600000,0.437500,0.889483,0.037433,0.437500
1,b_mask2,0.850796,0.925134,0.382353,0.062389,0.012478,0.013645,0.650000,0.270833,0.901961,0.023173,0.270833
2,b_mm,0.844731,0.923203,0.433735,0.068627,0.008170,0.009058,0.782609,0.300000,0.893791,0.029412,0.300000
3,balance_o_download,0.840459,0.921569,0.368421,0.075163,0.003268,0.003623,0.875000,0.233333,0.898693,0.022876,0.233333
4,balance_b_download,0.835326,0.918301,0.500000,0.057190,0.024510,0.027174,0.625000,0.416667,0.877451,0.040850,0.416667
5,o_stack,0.830631,0.910131,0.421053,0.065359,0.024510,0.027174,0.571429,0.333333,0.877451,0.032680,0.333333
6,balance_b_download2,0.819746,0.919935,0.436782,0.066993,0.013072,0.014493,0.703704,0.316667,0.888889,0.031046,0.316667
7,b_stack2,0.814191,0.913399,0.495238,0.055556,0.031046,0.034420,0.577778,0.433333,0.870915,0.042484,0.433333
8,balance_o_download2,0.808137,0.924837,0.477273,0.063725,0.011438,0.012681,0.750000,0.350000,0.890523,0.034314,0.350000
9,balance_w_download,0.807820,0.918301,0.479167,0.060458,0.021242,0.023551,0.638889,0.383333,0.880719,0.037582,0.383333


In [15]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
from collections import defaultdict

all_experiments = []

for base in [Path("../outputs4")]:
    for exp_name in os.listdir(base):
        exp_path = base / exp_name
        if not exp_path.is_dir():
            continue

        params_path  = exp_path / "params.json"

        for run_name in os.listdir(exp_path):
            run_path = exp_path / run_name
            if not run_path.is_dir():
                continue

            model_path   = run_path / "best_model.pth"
            results_path = run_path / "test_results.json"

            if not results_path.exists():
                continue

            with open(params_path, "r") as f:
                params = json.load(f)

            with open(results_path, "r") as f:
                results = json.load(f)

            entry = {
                "exp": exp_name,
                "run": run_name,
                "auc": float(results["auc"]),
            }

            # Add confusion-matrix-derived metrics
            for k, v in get_info(np.array(results["val_cfvalues"])).items():
                entry[k] = v

            all_experiments.append(entry)


In [17]:
df = pd.DataFrame(all_experiments)

# Convert the meta columns into index
df = df.set_index(["exp", "run"]).sort_index()

df


auc        tn        fp        fn        tp       tpr  \
exp       run                                                                
temp      run0  0.733999  0.878788  0.035651  0.067736  0.017825  0.208333   
          run1  0.649509  0.864528  0.049911  0.073084  0.012478  0.145833   
          run2  0.720110  0.903743  0.010695  0.073084  0.012478  0.145833   
temp copy run0  0.733999  0.878788  0.035651  0.067736  0.017825  0.208333   
          run1  0.649509  0.864528  0.049911  0.073084  0.012478  0.145833   
          run2  0.720110  0.903743  0.010695  0.073084  0.012478  0.145833   

                     fpr      prec    recall        f1       acc  
exp       run                                                     
temp      run0  0.038986  0.333333  0.208333  0.256410  0.896613  
          run1  0.054581  0.200000  0.145833  0.168675  0.877005  
          run2  0.011696  0.538462  0.145833  0.229508  0.916221  
temp copy run0  0.038986  0.333333  0.208333  0.256410  0.896613  
          run1  0.054581  0.200000  0.145833  0.168675  0.877005  
          run2  0.011696  0.538462  0.145833  0.229508  0.916221